In [4]:
import cv2
import numpy as np

In [5]:
# Videoquelle: 0 = Webcam, oder Dateipfad einsetzen
cap = cv2.VideoCapture(0)

In [6]:
# Hintergrundsubtraktor
bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50)


In [7]:
# Trajektorien speichern
tracks = []
max_trace_length = 30

In [8]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Resize für Performance
    frame = cv2.resize(frame, (640, 480))

    # Graustufen + Blur
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)

    # Hintergrundsubtraktion
    fg_mask = bg_subtractor.apply(gray)

    # Threshold
    _, thresh = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

    # Morphologische Filter
    kernel = np.ones((5,5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Konturen finden
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    centers = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 500:  # kleine Objekte ignorieren
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        # Bounding Box zeichnen
        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)

        # Mittelpunkt berechnen
        cx = int(x + w/2)
        cy = int(y + h/2)
        centers.append((cx, cy))

        cv2.circle(frame, (cx, cy), 4, (0,0,255), -1)

    # Tracking: einfache Trajektorien
    for center in centers:
        tracks.append([center])

    # Verlauf begrenzen
    if len(tracks) > 50:
        tracks = tracks[-50:]

    # Trajektorien zeichnen
    for track in tracks:
        for i in range(1, len(track)):
            cv2.line(frame, track[i-1], track[i], (255,0,0), 2)

    # Anzeige
    cv2.imshow("Frame", frame)
    cv2.imshow("Bewegungsmaske", thresh)

    key = cv2.waitKey(30)
    if key == 27:  # ESC
        break

In [9]:
cap.release()
cv2.destroyAllWindows()